# NLU project: train and evaluate on Google Colab

This notebook runs the four parts of the project on a Colab GPU.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. Every part can be trained on its own: run only the cells you need (`--exp <name>` trains a single experiment).
3. After every part, the checkpoints (`bin/`), the loss curves (`plots/`) and the logs are copied to Google Drive,
   so a disconnection does not lose the parts that are already finished.

In [ ]:
import torch
print("PyTorch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: Runtime > Change runtime type > T4 GPU")

## Setup

In [ ]:
import os

REPO = "https://github.com/giampyhatesyou/Natural_Language_Processing.git"
BRANCH = "main"  # branch with the code to run
ROOT = "/content/nlp"

if not os.path.isdir(ROOT):
    !git clone -q --depth 1 --branch {BRANCH} {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
%pip install -q -r {ROOT}/requirements.txt

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")

PARTS = ["language_modeling/part_A", "language_modeling/part_B",
         "intent_slot_filling/part_A", "intent_slot_filling/part_B"]
LOGS = f"{ROOT}/logs"
DRIVE_DIR = "/content/drive/MyDrive/nlu_project"  # where the results are copied
os.makedirs(LOGS, exist_ok=True)


def save_to_drive(part):
    """Copy bin/ and plots/ of a part, plus all the logs, to Google Drive."""
    if not os.path.isdir("/content/drive/MyDrive"):
        print("Google Drive is not mounted: the results stay only in this runtime")
        return
    for sub in ("bin", "plots"):
        src = os.path.join(ROOT, part, sub)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join(DRIVE_DIR, part, sub), dirs_exist_ok=True)
    shutil.copytree(LOGS, os.path.join(DRIVE_DIR, "logs"), dirs_exist_ok=True)
    print(f"{part}: results copied to {DRIVE_DIR}/{part}")

## LM part A: LSTM, dropout, AdamW

In [ ]:
!python -u {ROOT}/language_modeling/part_A/main.py --train 2>&1 | tee {LOGS}/lm_part_A.log
save_to_drive("language_modeling/part_A")

## LM part B: weight tying, variational dropout, NT-AvSGD

The longest part: after the trigger, NT-AvSGD keeps training while the averaged weights improve on dev.

In [ ]:
!python -u {ROOT}/language_modeling/part_B/main.py --train 2>&1 | tee {LOGS}/lm_part_B.log
save_to_drive("language_modeling/part_B")

## NLU part A: LSTM, BiLSTM, BiLSTM + dropout (5 runs each)

In [ ]:
!python -u {ROOT}/intent_slot_filling/part_A/main.py --train 2>&1 | tee {LOGS}/nlu_part_A.log
save_to_drive("intent_slot_filling/part_A")

## NLU part B: BERT (5 runs)

The checkpoint of the best run is about 420 MB.

In [ ]:
!python -u {ROOT}/intent_slot_filling/part_B/main.py --train 2>&1 | tee {LOGS}/nlu_part_B.log
save_to_drive("intent_slot_filling/part_B")

## Restore the checkpoints from Drive (optional)

In a new runtime: run the setup cells, then this one, to evaluate the saved models without training them again.

In [ ]:
for part in PARTS:
    src = os.path.join(DRIVE_DIR, part, "bin")
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(ROOT, part, "bin"), dirs_exist_ok=True)
        print("restored", part)

## Evaluate the checkpoints on the test sets

Without `--train`, every `main.py` loads the checkpoints in its `bin/` folder and evaluates them.

In [ ]:
for part in PARTS:
    print(f"===== {part}")
    !python {ROOT}/{part}/main.py 2>&1 | grep -v "it/s"

## Download the exam folder

Code, datasets, checkpoints and plots in the structure required for the course submission:
`258237-Giampietro-Andrea/LM` and `258237-Giampietro-Andrea/NLU`.

In [ ]:
STAGE = "/content/exam_zip/258237-Giampietro-Andrea"
shutil.rmtree("/content/exam_zip", ignore_errors=True)
skip = shutil.ignore_patterns("__pycache__")
shutil.copytree(f"{ROOT}/language_modeling", f"{STAGE}/LM", ignore=skip)
shutil.copytree(f"{ROOT}/intent_slot_filling", f"{STAGE}/NLU", ignore=skip)
archive = shutil.make_archive("/content/258237-Giampietro-Andrea", "zip", "/content/exam_zip")

from google.colab import files
files.download(archive)